# ◆ LeadHarbour: from a score to a decision

Run `leadharbour fetch` first, from the repository root. The official UCI data describes bank contacts from 2008 to 2010. It is useful for learning, not for choosing current recipients. The outcome is a response label, not a causal uplift label.

In [ ]:
from pathlib import Path
from leadharbour.data import FEATURES, FORBIDDEN_FEATURES, load_dataset
from leadharbour.model import ordered_split, train

source = Path('data/bank-additional-full.csv')
frame = load_dataset(source)
training, validation, test = ordered_split(frame)
print('Rows:', len(frame), 'Features:', FEATURES)
print('Outcome rates:', [round(part.y.eq('yes').mean(), 3) for part in (training, validation, test)])
assert set(FEATURES).isdisjoint(FORBIDDEN_FEATURES)


The original rows are ordered by date. Keeping a later period for testing exposes a change in outcome rate that a random split could hide. Call duration is excluded because it is known only after a call. Age and several demographic fields are also excluded from the baseline, though the remaining fields still need a fairness review.

In [ ]:
report = train(source, Path('artifacts/model.joblib'))
print('Validation:', report['validation'])
print('Later test period:', report['test'])
assert report['test']['rows'] > 0


A ranking measure is not enough. Compare average precision with the outcome rate, inspect the Brier score, and ask whether the probability means the same thing in a later period. The model card records the source hash and split so a result can be reproduced.

In [ ]:
from leadharbour.decision import Candidate, expected_net_value, rank_for_review

synthetic = [Candidate('example-a', 0.20, 100, 5),
             Candidate('example-b', 0.05, 100, 5)]
print([(item.lead_id, expected_net_value(item)) for item in synthetic])
print('For human review:', [item.lead_id for item in rank_for_review(synthetic, 1)])


Expected net value is `probability × value if converted − contact cost`. The numbers above are invented to explain the arithmetic. A response probability does not tell us the extra conversions caused by contact. For that, we need a credible treatment and control comparison, with consent and business rules checked separately.